<a href="https://colab.research.google.com/github/MatheusRPessoa/fine-tunings/blob/main/fine_tunning_atendimento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
%%capture
!pip install transformers
!pip install datasets
!pip install evaluate

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
import os

In [ ]:
dataset = load_dataset("json", data_files={"train": "/content/treino.jsonl", "test": "/content/teste.jsonl"})

In [ ]:
dataset

In [ ]:
checkpoint = 'neuralmind/bert-base-portuguese-cased'

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

mapDict = {
    "venda": 0,
    "suporte": 1,
}

def transform_labels(batch):
  return {"labels": [mapDict[c] for c in batch["completion"]]}

def tokenizer_function(batch):
  return tokenizer(batch["prompt"], truncate=True)

In [ ]:
tokenized_datasets = dataset.map(tokenizer_function, batched=True)
tokenized_datasets = tokenized_datasets.map(transform_labels, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["prompt", "completion"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import TrainingArguments

output_dir = "./bertimbau-intencao-atendimento"

training_args = TrainingArguments(
    output_dir = output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
)

In [ ]:
from transformers import Trainer, AutoModelForSequenceClassification

id2Label = {0: "venda", 1: "suporte"}
label2id = {"venda": 0, "suporte": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    'neuralmind/bert-base-portuguese-cased',
    num_labels = 2,
    id2label=id2Label,
    label2id=label2id,
)

os.environ['WANDB_DISABLE'] = "true"
os.environ['WANDB_MODE'] = "offline"

In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metric(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis= -1)
  return metric.compute(predictions=predictions, references=labels)

In [ ]:
trainer = Trainer(
    model,
    training_args,
    train_dataset = tokenized_datasets['train'],
    eval_dataset = tokenized_datasets['test'],
    data_collator = data_collator,
    processing_class = tokenizer,
    compute_metrics = compute_metric,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
trainer.save_model()

In [ ]:
trainer.push_to_hub("matheusrp41/modelservice")

In [ ]:
from transformers import pipeline

pipe = pipeline("text-classification", model="matheusrp41/bertimbau-intencao-atendimento")

In [ ]:
pipe("Quero comprar uma geladeira")